# 01 — Spark-Specific Extensions

Spark-focused helper libraries and built-in PySpark techniques for validation, data quality, profiling, testing, DataFrame comparison, and quick diagnostics.

## 00 — Spark setup
Standard SparkSession setup for the local cluster notebooks.

In [1]:
from pyspark.sql import SparkSession, functions as F
import time

spark = (
    SparkSession.builder
    .appName("spark_specific_extensions")
    .config("spark.sql.warehouse.dir", "/tmp/spark-warehouse")
    .getOrCreate()
)

spark.conf.set("spark.sql.shuffle.partitions", "64")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.sql.debug.maxToStringFields", "10000")

spark

## 01 — Extension map
This table keeps the Spark-specific tooling organized by purpose. This table keeps the Spark-specific tooling organized by purpose.

In [2]:
import pandas as pd
from importlib.util import find_spec

extensions = [
    ("quinn", "PySpark helper functions", "Column validation, schema helpers, string cleaning, schema-to-code"),
    ("chispa", "PySpark testing", "DataFrame equality and schema equality assertions"),
    ("cuallee", "Data quality checks", "Completeness, uniqueness, ranges, patterns, rule-based validation"),
    ("fg-data-profiling", "Profiling reports", "HTML profiling report after collecting a small Spark sample to Pandas"),
    ("pandera", "Schema validation", "Schema validation using Pandas DataFrame models and type annotations"),
    ("built-in PySpark", "Native diagnostics", "describe, summary, approxQuantile, null counts, distinct counts"),
]

pd.DataFrame(extensions, columns=["tool", "category", "what_it_does"])


,tool,category,what_it_does
0,quinn,PySpark helper functions,"Column validation, schema helpers, string clea..."
1,chispa,PySpark testing,DataFrame equality and schema equality assertions
2,cuallee,Data quality checks,"Completeness, uniqueness, ranges, patterns, ru..."
3,fg-data-profiling,Profiling reports,HTML profiling report after collecting a small...
4,pandera,Schema validation,Schema validation using Pandas DataFrame model...
5,built-in PySpark,Native diagnostics,"describe, summary, approxQuantile, null counts..."


## 02 — Package availability
Checks which Python packages are available in the current notebook container.

In [3]:
checks = [
    ("quinn", "quinn"),
    ("chispa", "chispa"),
    ("cuallee", "cuallee"),
    ("fg-data-profiling", "data_profiling"),
    ("pandera", "pandera"),
]

pd.DataFrame(
    [(tool, module, find_spec(module) is not None) for tool, module in checks],
    columns=["tool", "import_name", "available"]
)


,tool,import_name,available
0,quinn,quinn,True
1,chispa,chispa,True
2,cuallee,cuallee,True
3,fg-data-profiling,data_profiling,False
4,pandera,pandera,True


## 03 — Sample data
Small intentionally imperfect dataset used by the validation, quality, profiling, and diagnostics examples.

In [4]:
events = spark.createDataFrame(
    [
        (1, "u001", " view ", "SK", 19.90, "2026-01-01"),
        (2, "u002", "CLICK", "CZ", 29.90, "2026-01-01"),
        (3, "u003", "purchase", "DE", 99.90, "2026-01-02"),
        (4, "u003", "purchase", "DE", 99.90, "2026-01-02"),
        (5, None, "view", "SK", 9.90, "2026-01-03"),
        (6, "u006", None, "AT", -5.00, "2026-01-03"),
        (7, "u007", "search", None, 14.50, "2026-01-04"),
    ],
    ["event_id", "user_id", "event_type", "country", "amount", "event_date"],
)

events.show(truncate=False)
events.printSchema()

+--------+-------+----------+-------+------+----------+
|event_id|user_id|event_type|country|amount|event_date|
+--------+-------+----------+-------+------+----------+
|1       |u001   | view     |SK     |19.9  |2026-01-01|
|2       |u002   |CLICK     |CZ     |29.9  |2026-01-01|
|3       |u003   |purchase  |DE     |99.9  |2026-01-02|
|4       |u003   |purchase  |DE     |99.9  |2026-01-02|
|5       |NULL   |view      |SK     |9.9   |2026-01-03|
|6       |u006   |NULL      |AT     |-5.0  |2026-01-03|
|7       |u007   |search    |NULL   |14.5  |2026-01-04|
+--------+-------+----------+-------+------+----------+

root
 |-- event_id: long (nullable = true)
 |-- user_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- country: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- event_date: string (nullable = true)



## 04 — Built-in PySpark diagnostics
Native PySpark should be the default first step: it is always available, works on distributed data, and avoids extra package dependency risk.

In [5]:
events.describe("amount").show()
events.summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show()

null_counts = events.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in events.columns
])

null_counts.show(truncate=False)

approx = events.approxQuantile("amount", [0.25, 0.5, 0.75, 0.95], 0.01)
approx

+-------+-----------------+
|summary|           amount|
+-------+-----------------+
|  count|                7|
|   mean|38.42857142857143|
| stddev|43.29359899899423|
|    min|             -5.0|
|    max|             99.9|
+-------+-----------------+



+-------+-----------------+-------+----------+-------+-----------------+----------+
|summary|         event_id|user_id|event_type|country|           amount|event_date|
+-------+-----------------+-------+----------+-------+-----------------+----------+
|  count|                7|      6|         6|      6|                7|         7|
|   mean|              4.0|   NULL|      NULL|   NULL|38.42857142857143|      NULL|
| stddev|2.160246899469287|   NULL|      NULL|   NULL|43.29359899899423|      NULL|
|    min|                1|   u001|     view |     AT|             -5.0|2026-01-01|
|    25%|                2|   NULL|      NULL|   NULL|              9.9|      NULL|
|    50%|                4|   NULL|      NULL|   NULL|             19.9|      NULL|
|    75%|                6|   NULL|      NULL|   NULL|             99.9|      NULL|
|    max|                7|   u007|      view|     SK|             99.9|2026-01-04|
+-------+-----------------+-------+----------+-------+-----------------+----

[9.9, 19.9, 99.9, 99.9]

## 05 — quinn
Quinn is useful for small PySpark helper functions: validating column presence, cleaning string columns, inspecting schemas, and generating reusable schema/code snippets.

In [6]:
required_columns = {"event_id", "user_id", "event_type", "country", "amount", "event_date"}
missing_columns = required_columns - set(events.columns)

if missing_columns:
    raise ValueError(f"Missing columns: {sorted(missing_columns)}")

clean_events = events.select(
    "event_id",
    "user_id",
    F.lower(F.trim(F.col("event_type"))).alias("event_type"),
    F.upper(F.trim(F.col("country"))).alias("country"),
    "amount",
    F.to_date("event_date").alias("event_date"),
)

try:
    import quinn
    print("quinn imported")
except Exception as exc:
    print("quinn not available; using equivalent built-in PySpark transformations")
    print(type(exc).__name__, exc)

clean_events.show(truncate=False)

quinn imported


[Stage 12:>                                                         (0 + 1) / 1]

+--------+-------+----------+-------+------+----------+
|event_id|user_id|event_type|country|amount|event_date|
+--------+-------+----------+-------+------+----------+
|1       |u001   |view      |SK     |19.9  |2026-01-01|
|2       |u002   |click     |CZ     |29.9  |2026-01-01|
|3       |u003   |purchase  |DE     |99.9  |2026-01-02|
|4       |u003   |purchase  |DE     |99.9  |2026-01-02|
|5       |NULL   |view      |SK     |9.9   |2026-01-03|
|6       |u006   |NULL      |AT     |-5.0  |2026-01-03|
|7       |u007   |search    |NULL   |14.5  |2026-01-04|
+--------+-------+----------+-------+------+----------+



## 06 — chispa
Chispa is useful in tests. It compares DataFrames and schemas with readable assertion failures, which is better than manually comparing collected rows.

In [7]:
expected = spark.createDataFrame(
    [(1, "u001", "view", "SK", 19.90)],
    ["event_id", "user_id", "event_type", "country", "amount"],
)

actual = clean_events.filter(F.col("event_id") == 1).select("event_id", "user_id", "event_type", "country", "amount")

try:
    from chispa.dataframe_comparer import assert_df_equality
    assert_df_equality(actual, expected, ignore_nullable=True)
    print("chispa assertion passed")
except Exception as exc:
    print("chispa not available or assertion failed; fallback comparison below")
    print(type(exc).__name__, exc)
    print(actual.collect() == expected.collect())

chispa assertion passed


## 07 — cuallee
Cuallee is useful for declarative data quality rules. It lets you express checks such as completeness, uniqueness, ranges, and allowed values in a compact way.

In [8]:
try:
    from cuallee import Check, CheckLevel

    check = (
        Check(CheckLevel.WARNING, "events_quality")
        .is_complete("event_id")
        .is_complete("user_id")
        .is_unique("event_id")
        .is_greater_than("amount", 0)
        .is_contained_in("country", ("SK", "CZ", "DE", "AT", "PL"))
    )

    result = check.validate(clean_events)
    display(result)
except Exception as exc:
    print("cuallee not available; using equivalent built-in PySpark checks")
    print(type(exc).__name__, exc)
    clean_events.agg(
        F.count("event_id").alias("event_id_complete"),
        F.count("user_id").alias("user_id_complete"),
        F.countDistinct("event_id").alias("event_id_distinct"),
        F.sum(F.when(F.col("amount") <= 0, 1).otherwise(0)).alias("amount_not_positive"),
        F.sum(F.when(~F.col("country").isin("SK", "CZ", "DE", "AT", "PL") | F.col("country").isNull(), 1).otherwise(0)).alias("invalid_country"),
    ).show(truncate=False)

DataFrame[id: int, timestamp: string, check: string, level: string, column: string, rule: string, value: string, rows: bigint, violations: bigint, pass_rate: double, pass_threshold: double, status: string]

## 08 — fg-data-profiling
FG Data Profiling is useful for HTML profiling reports. Use it only after reducing or sampling Spark data into a small Pandas DataFrame.

In [9]:
sample_pdf = clean_events.limit(1000).toPandas()

try:
    from data_profiling import ProfileReport
    profile = ProfileReport(sample_pdf, title="Events profiling", minimal=True)
    profile.to_file("/tmp/events_profile.html")
    print("Profile written to /tmp/events_profile.html")
except Exception as exc:
    print("fg-data-profiling not available")
    print(type(exc).__name__, exc)
    display(sample_pdf.describe(include="all"))

fg-data-profiling not available
ModuleNotFoundError No module named 'data_profiling'


,event_id,user_id,event_type,country,amount,event_date
count,7.000000,6,6,6,7.000000,7
unique,NaN,5,4,4,NaN,4
top,NaN,u003,view,SK,NaN,2026-01-01
freq,NaN,2,2,2,NaN,2
mean,4.000000,NaN,NaN,NaN,38.428571,NaN
std,2.160247,NaN,NaN,NaN,43.293599,NaN
min,1.000000,NaN,NaN,NaN,-5.000000,NaN
25%,2.500000,NaN,NaN,NaN,12.200000,NaN
50%,4.000000,NaN,NaN,NaN,19.900000,NaN
75%,5.500000,NaN,NaN,NaN,64.900000,NaN


## 09 — pandera
Pandera is useful for schema validation with Python type annotations. For Spark-scale data, use it on small Pandas samples or targeted extracts unless you intentionally use Pandera's PySpark integration.

In [10]:
try:
    import pandera.pandas as pa

    schema = pa.DataFrameSchema(
        {
            "event_id": pa.Column(int, pa.Check.ge(1)),
            "user_id": pa.Column(object, nullable=True),
            "event_type": pa.Column(object, nullable=True),
            "country": pa.Column(object, nullable=True),
            "amount": pa.Column(float),
            "event_date": pa.Column(object),
        }
    )

    schema.validate(sample_pdf, lazy=True)
    print("pandera validation passed")
except Exception as exc:
    print("pandera not available or validation failed")
    print(type(exc).__name__, exc)


pandera validation passed


## 10 — Decision table
Final practical mapping from Spark notebook task to the preferred tool.